# Card Fraud Triage: GMM vs Deep Autoencoder Anomaly Detection

## Project 26: The 'Hybrid' Logic: Gaussian Density Estimation to Learned Nonlinear Reconstruction

### Abstract & Overview
In credit card fraud monitoring, fraud cases constitute a minute fraction (~0.17%) of all transactions. Supervised classifiers trained on rare labels are prone to severe overfitting, sensitivity to concept drift, and vulnerability to emerging fraud schemes.

This notebook demonstrates and audits a **two-pronged unsupervised anomaly-ranking triage system**:
1. **AML Baseline (Gaussian Mixture Density Scoring)**: Learns parametric density clusters $p(x)$ on legitimate transactions using the EM algorithm. Low log-likelihood is the anomaly score.
2. **Deep Learning Model (Undercomplete Autoencoder)**: Compresses legitimate transactions into a low-dimensional bottleneck ($29 \to 20 \to 12 \to 6 \to 12 \to 20 \to 29$) and reconstructs them. Per-sample **Mean Squared Error (MSE)** reconstruction loss is the anomaly score.
3. **Fair Comparison**: Both models are trained **strictly on the identical 169,893 legitimate training transactions** (chronological 60% split) with features scaled by a legitimate-only `StandardScaler`. Neither model sees fraud transactions or test transactions during training or threshold calibration.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)
import torch

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLEAN_PATH, ARTIFACTS, SEED, TARGET_FALSE_POSITIVE_RATE
from src.autoencoder import TransactionAutoencoder

print(f"PyTorch version: {torch.__version__}")
print(f"Artifacts directory: {ARTIFACTS}")

## 1. Load Chronological Data Partitions

Following strict temporal holdout principles, the cleaned dataset (283,726 rows) is split chronologically:
- **Train (60%)**: 170,235 rows (169,893 legitimate, 342 fraud)
- **Validation (20%)**: 56,745 rows (56,688 legitimate, 57 fraud)
- **Test (20%)**: 56,746 rows (56,672 legitimate, 74 fraud)

In [ ]:
data = pd.read_csv(CLEAN_PATH).sort_values("Time").reset_index(drop=True)
train_end = int(len(data) * 0.60)
val_end = int(len(data) * 0.80)

train_df = data.iloc[:train_end].copy()
val_df = data.iloc[train_end:val_end].copy()
test_df = data.iloc[val_end:].copy()

features = [col for col in data.columns if col not in ("Time", "Class")]
print(f"Feature count: {len(features)}")
print(f"Train legitimate: {(train_df['Class'] == 0).sum():,} | Frauds: {train_df['Class'].sum():,}")
print(f"Val legitimate:   {(val_df['Class'] == 0).sum():,} | Frauds: {val_df['Class'].sum():,}")
print(f"Test legitimate:  {(test_df['Class'] == 0).sum():,} | Frauds: {test_df['Class'].sum():,}")

## 2. Load Audited Model Artifacts

We load both the saved GMM audit report & bundle, and the Deep Autoencoder bundle.

In [ ]:
# Load GMM report and bundle
with open(ARTIFACTS / "gmm_report.json") as f:
    gmm_report = json.load(f)

# Load Autoencoder report and bundle
with open(ARTIFACTS / "autoencoder_report.json") as f:
    ae_report = json.load(f)

ae_bundle = joblib.load(ARTIFACTS / "autoencoder_bundle.joblib")
print("Loaded Autoencoder and GMM audit reports successfully.")

## 3. Score Test Transactions with Autoencoder

Reconstruction error is calculated using the fitted `StandardScaler` and `TransactionAutoencoder` weights.

In [ ]:
# Instantiate model and load trained state
ae_model = TransactionAutoencoder(
    input_dim=ae_bundle["model_config"]["input_dim"],
    hidden_dims=tuple(ae_bundle["model_config"]["hidden_dims"]),
    latent_dim=ae_bundle["model_config"]["latent_dim"],
)
ae_model.load_state_dict(ae_bundle["model_state_dict"])
ae_model.eval()

scaler = ae_bundle["scaler"]
test_scaled = scaler.transform(test_df[features])
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

with torch.no_grad():
    ae_test_scores = ae_model.compute_reconstruction_error(test_tensor).numpy()

test_y = test_df["Class"].to_numpy()
print(f"Autoencoder Test PR-AUC:  {average_precision_score(test_y, ae_test_scores):.4f}")
print(f"Autoencoder Test ROC-AUC: {roc_auc_score(test_y, ae_test_scores):.4f}")

## 4. Head-to-Head Precision-Recall and ROC Comparison

Comparing the GMM negative log-likelihood density score against the Autoencoder nonlinear reconstruction score.

In [ ]:
# GMM test metrics from audit log
gmm_test_metrics = gmm_report["test_evaluation"]
print("=== GMM vs Autoencoder Benchmark ===")
print(f"Metric                  GMM (Full Covar)      Autoencoder (AE)")
print(f"----------------------------------------------------------------")
print(f"Average Precision (AP)  {gmm_test_metrics['average_precision']:.4f}                {ae_report['test_metrics']['average_precision']:.4f}")
print(f"ROC-AUC                 {gmm_test_metrics['roc_auc']:.4f}                {ae_report['test_metrics']['roc_auc']:.4f}")
print(f"Recall @ 1% Val FPR     {gmm_test_metrics['recall']*100:.2f}%               {ae_report['test_metrics']['recall']*100:.2f}%")
print(f"Precision @ 1% Val FPR  {gmm_test_metrics['precision']*100:.2f}%               {ae_report['test_metrics']['precision']*100:.2f}%")
print(f"Frauds Detected (/74)   {gmm_test_metrics['frauds_detected']}                    {ae_report['test_metrics']['frauds_detected']}")
print(f"False Alarms            {gmm_test_metrics['false_positives']}                   {ae_report['test_metrics']['false_alarms']}")

## 5. Review Capacity Trade-Off (Precision@K and Recall@K)

In operational triage, investigators review a fixed queue of the highest-scoring alerts each day.

In [ ]:
ae_capacity = ae_report["capacity_metrics"]
gmm_capacity = gmm_report["test_evaluation"]["precision_at_capacities"]

capacity_df = pd.DataFrame([
    {
        "Capacity (K)": 100,
        "GMM Frauds": gmm_capacity["top_100"]["frauds_detected"],
        "GMM Precision@K": f"{gmm_capacity['top_100']['precision_at_k']*100:.1f}%",
        "AE Frauds": ae_capacity["top_100"]["frauds_detected"],
        "AE Precision@K": f"{ae_capacity['top_100']['precision_at_k']*100:.1f}%",
    },
    {
        "Capacity (K)": 500,
        "GMM Frauds": gmm_capacity["top_500"]["frauds_detected"],
        "GMM Precision@K": f"{gmm_capacity['top_500']['precision_at_k']*100:.1f}%",
        "AE Frauds": ae_capacity["top_500"]["frauds_detected"],
        "AE Precision@K": f"{ae_capacity['top_500']['precision_at_k']*100:.1f}%",
    },
    {
        "Capacity (K)": 1000,
        "GMM Frauds": gmm_capacity["top_1000"]["frauds_detected"],
        "GMM Precision@K": f"{gmm_capacity['top_1000']['precision_at_k']*100:.1f}%",
        "AE Frauds": ae_capacity["top_1000"]["frauds_detected"],
        "AE Precision@K": f"{ae_capacity['top_1000']['precision_at_k']*100:.1f}%",
    },
])
capacity_df